<a href="https://colab.research.google.com/github/AdelineKwakye/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print("Total revenue is ", total_revenue)
print("Total units is ", total_units)

Total revenue is  8520.0
Total units is  783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
rev_by_category =df.groupby('category').agg(
    revenue = ('revenue', 'sum')
).sort_values('revenue', ascending = False)

rev_by_category['share'] = rev_by_category['revenue'] / rev_by_category['revenue'].sum() * 100
print("Revenue sorted by category: \n")
rev_by_category


Revenue sorted by category: 



,revenue,share
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
vendor_avg = df.groupby ('vendor_id').agg(
    average_revenue = ('revenue', 'mean'),
    order_count = ('revenue', 'count'),
).sort_values('average_revenue', ascending=False)
print(vendor_avg)
highest_vendor = vendor_avg.iloc[0]

print("\nThe vendor with the highest average order revenue was: ", highest_vendor.name)
for vendor, row in vendor_avg.iterrows():
    print(f"{vendor} had {row['order_count']} orders.")

           average_revenue  order_count
vendor_id                              
V-01             22.595745           94
V-18             21.750000          108
V-05             20.580645           93
V-10             20.314286          105

The vendor with the highest average order revenue was:  V-01
V-01 had 94.0 orders.
V-18 had 108.0 orders.
V-05 had 93.0 orders.
V-10 had 105.0 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO

by_category =df.groupby('category').agg(
    revenue = ('revenue', 'sum')
).sort_values('revenue', ascending = False)

by_category['share'] = by_category['revenue'] / by_category['revenue'].sum() * 100

by_category.loc['Merch']
print(f"The share of revenue that comes from Merch is: {by_category.loc['Merch', 'share']:.1f}%")

The share of revenue that comes from Merch is: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

original_rows = len(df)
original_revene = df['revenue'].sum()

joined = df.merge(vendor_names, on="vendor_id", how="left", indicator='many-to-one')
print("Merged df number of rows: ", len(joined))
print("original df number of rows: ", original_rows)

print("The missing vendor is: ", joined[joined['vendor_name'].isna()]['vendor_id'].unique())

joined['vendor_name'] = joined['vendor_name'].fillna("Unknown")

Merged df number of rows:  400
original df number of rows:  400
The missing vendor is:  ['V-18']


**The unmatched vendor, and what I did about it:** The unknown vendor is V-18 and I decided to just fill in the the na with unknown. This way, the orders are still considered valid and we just acknowledge the fact the specific vendor name is not known.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
revenue_pivot = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum'

)
revenue_pivot
print("Revenue pivot table: ")

Revenue pivot table: 


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

(a) Someting that I would tell the vendors, specifically for Hoos Burgers, would be to try and get the drink sales up a little bit more. In the table we can see that food makes then 1,338 dollars in revenue but the drinks are only giving them 171. So to raise that number maybe they could try and do a combo meal were the drink comes with the food to maybe get that drink revenue up there as well as what they're making from food. Another thing I would suggest for all the vendors would be to invest more towards food and less towards rain gear. Food contributes about 50.4% of the shares while rain gear only makes 10.6%. It seems as if money is being wasted towards rain gear when it would be more profitable to focus on food.

(b) I would say Answer 5 because, I decided to label the missing vendor name as Unknown. Vendor V-18 was seen to be the second highest contributor to revenue, so having that vendor be filled in as Unknown leaves a big blindspot because we don't know who that vendor is.